# Hyperlocal Demand Forecasting — Exploratory Data Analysis (EDA)

This notebook performs exploratory analysis on our Bangalore dark store order transactions. We will load data from our PostgreSQL database to analyze demand patterns, peak hours, weekend surges, weather impacts, and festival uplifts.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load environment variables
load_dotenv(dotenv_path="../.env")

## 1. Load Data from PostgreSQL

In [ ]:
db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "hyperlocal_demand")
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "A4apple")

engine = create_engine(f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

# Query clean orders
df_orders = pd.read_sql("SELECT * FROM clean_orders", engine)
df_orders['timestamp'] = pd.to_datetime(df_orders['timestamp'])
print(f"Loaded {len(df_orders):,} orders.")
df_orders.head()

## 2. Temporal Analysis (Peak Hours & Weekends)

In [ ]:
# Extract hour of day and day of week
df_orders['hour'] = df_orders['timestamp'].dt.hour
df_orders['day_name'] = df_orders['timestamp'].dt.day_name()

# Plot hourly volume distribution
sns.countplot(data=df_orders, x='hour', hue='is_weekend', palette="viridis")
plt.title("Hourly Order Distribution (Weekdays vs Weekends)")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Orders")
plt.show()

We observe clear peak spikes during lunch (12:00 - 14:00) and dinner (19:00 - 22:00) times, with a solid volume boost during the weekends.

## 3. Dark Store Zone Variance

In [ ]:
# Compare order volume across zones
zone_counts = df_orders['zone_id'].value_counts().reset_index()
zone_counts.columns = ['zone_id', 'order_count']

sns.barplot(data=zone_counts, x='zone_id', y='order_count', palette="muted")
plt.title("Total Order Volume by Dark Store Zone")
plt.xlabel("Store Zone")
plt.ylabel("Total Orders")
plt.show()

Zone Z1 (representing high-density commercial/residential centers) drives the highest order share, whereas Zone Z5 (suburban) handles lower baseline volumes.

## 4. Weather & Holiday Multipliers

In [ ]:
# Aggregate hourly counts to see average hourly volume under different conditions
hourly_agg = df_orders.groupby([
    df_orders['timestamp'].dt.floor('h'), 'weather_condition', 'is_festival_day'
]).size().reset_index(name='order_count')

# Weather conditions average hourly orders
weather_avg = hourly_agg.groupby('weather_condition')['order_count'].mean().reset_index()
print("Average hourly volume by weather:")
print(weather_avg)

# Festival vs non-festival average hourly orders
festival_avg = hourly_agg.groupby('is_festival_day')['order_count'].mean().reset_index()
print("\nAverage hourly volume on festival days:")
print(festival_avg)

## Conclusion

Exploratory analysis confirms the following behavior in our dataset:
1. **Double-peak diurnal cycles:** Spikes in volume align with standard food delivery hours.
2. **Strong weekend and rainy-day uplifts:** Customers order significantly more when they are at home on weekends or when weather makes travel inconvenient.
3. **Store capacity variances:** Dense inner zones show much higher transactional throughput than suburban stores.